# Hotel Booking Dataset — Data Cleaning Pipeline
**Role:** Junior Data Scientist  
**Dataset:** 55,949 entries × 49 columns  
**Objective:** Clean the raw dataset and export a production-ready CSV for downstream EDA / modelling.

---
### Cleaning Roadmap
| Step | Task |
|------|------|
| 1 | Load dataset & quick sanity check |
| 2 | Standardise column names |
| 3 | Fix data types |
| 4 | Handle missing values |
| 5 | Remove duplicates |
| 6 | Outlier detection & capping |
| 7 | Fix inconsistent / invalid values |
| 8 | Final quality report |
| 9 | Export cleaned CSV |

---
## Step 1 · Load Dataset & Sanity Check

In [49]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.2f}'.format)
print('✅ Libraries loaded.')

✅ Libraries loaded.


In [50]:
# ── Load raw data ─────────────────────────────────────────────────────────────
# 🔧 Update this path to your file location
RAW_PATH     = '/Users/musthyalasadhvik/Desktop/SectionB_G-1_StayWise/data/raw/hotel_bookings_Raw.csv'        # <-- change me
CLEANED_PATH = '/Users/musthyalasadhvik/Desktop/SectionB_G-1_StayWise/data/processed/hotel_bookings_cleaned.csv'

df_raw = pd.read_csv(RAW_PATH)

# Keep a snapshot of the original for comparison at the end
original_shape = df_raw.shape

print(f'Raw shape  : {df_raw.shape[0]:,} rows  ×  {df_raw.shape[1]} columns')
print()
df_raw.head(3)

Raw shape  : 119,209 rows  ×  49 columns



,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,average_daily_rate,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,meal_type,country_name,total_nights,total_guests,total_revenue,has_children,is_weekend_stay,has_special_requests,is_direct_booking,lead_time_category,stay_duration_category,price_tier,customer_segment,booking_type,season,is_peak_season,revenue_per_guest,average_night_value,clv_proxy
0,Resort Hotel,0,342.00,2015,July,27,1,0,0,2,0.00,0,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,0,0,Transient,0.00,0,0,Check-Out,2015-07-01,Bed & Breakfast,Portugal,0,2.00,0.00,0,0,0,1,Very Long Term (180d+),NaN,Budget,Non-Family,Standard Booking,Summer,1,0.00,0.00,0.00
1,Resort Hotel,0,444.00,2015,July,27,1,0,0,2,0.00,0,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,0,0,Transient,0.00,0,0,Check-Out,2015-07-01,Bed & Breakfast,Portugal,0,2.00,0.00,0,0,0,1,Very Long Term (180d+),NaN,Budget,Non-Family,Standard Booking,Summer,1,0.00,0.00,0.00
2,Resort Hotel,0,7.00,2015,July,27,1,0,1,1,0.00,0,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,0,0,Transient,75.00,0,0,Check-Out,2015-07-02,Bed & Breakfast,United Kingdom,1,1.00,75.00,0,0,0,1,Last Minute (0-7d),1 Night,Economy,Non-Family,Quick Trip,Summer,1,75.00,75.00,75.00


In [51]:
# ── Quick dtype overview ──────────────────────────────────────────────────────
print('Column dtypes:')
print(df_raw.dtypes.value_counts())
print()
print('Sample of all columns:')
display(df_raw.describe(include='all').T)

Column dtypes:
int64      22
str        19
float64     8
Name: count, dtype: int64

Sample of all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
hotel,119209,2,City Hotel,79163,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_canceled,119209.00,NaN,NaN,NaN,0.37,0.48,0.00,0.00,0.00,1.00,1.00
lead_time,119209.00,NaN,NaN,NaN,103.51,104.61,0.00,18.00,69.00,161.00,444.00
arrival_date_year,119209.00,NaN,NaN,NaN,2016.16,0.71,2015.00,2016.00,2016.00,2017.00,2017.00
arrival_date_month,119209,12,August,13861,NaN,NaN,NaN,NaN,NaN,NaN,NaN
arrival_date_week_number,119209.00,NaN,NaN,NaN,27.16,13.60,1.00,16.00,28.00,38.00,53.00
arrival_date_day_of_month,119209.00,NaN,NaN,NaN,15.80,8.78,1.00,8.00,16.00,23.00,31.00
stays_in_weekend_nights,119209.00,NaN,NaN,NaN,0.93,1.00,0.00,0.00,1.00,2.00,19.00
stays_in_week_nights,119209.00,NaN,NaN,NaN,2.50,1.90,0.00,1.00,2.00,3.00,50.00
adults,119209.00,NaN,NaN,NaN,1.86,0.58,0.00,2.00,2.00,2.00,55.00


---
## Step 2 · Standardise Column Names

In [52]:
# ── Work on a copy — never mutate the raw dataframe ──────────────────────────
df = df_raw.copy()

# Lowercase, strip whitespace, replace spaces/hyphens with underscores
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-]+', '_', regex=True)   # spaces & hyphens → _
    .str.replace(r'[^\w]', '', regex=True)       # remove any other special chars
)

print('✅ Column names standardised.')
print('Columns:', df.columns.tolist())

✅ Column names standardised.
Columns: ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'days_in_waiting_list', 'customer_type', 'average_daily_rate', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date', 'meal_type', 'country_name', 'total_nights', 'total_guests', 'total_revenue', 'has_children', 'is_weekend_stay', 'has_special_requests', 'is_direct_booking', 'lead_time_category', 'stay_duration_category', 'price_tier', 'customer_segment', 'booking_type', 'season', 'is_peak_season', 'revenue_per_guest', 'average_night_value', 'clv_proxy']


---
## Step 3 · Fix Data Types

In [53]:
# ── 3a. Identify columns that should be categorical ───────────────────────────
#
# Rule: object columns with fewer than 30 unique values are good candidates
# for category dtype (saves memory, faster groupby operations)

LOW_CARDINALITY_THRESHOLD = 30

object_cols = df.select_dtypes(include='object').columns.tolist()
cat_candidates = [
    col for col in object_cols
    if df[col].nunique() <= LOW_CARDINALITY_THRESHOLD
]

print(f'Object columns          : {len(object_cols)}')
print(f'→ Category candidates   : {len(cat_candidates)}')
print(f'→ High-cardinality text : {len(object_cols) - len(cat_candidates)}')
print()
print('Category candidates:', cat_candidates)

Object columns          : 19
→ Category candidates   : 17
→ High-cardinality text : 2

Category candidates: ['hotel', 'arrival_date_month', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'reservation_status', 'meal_type', 'country_name', 'lead_time_category', 'stay_duration_category', 'price_tier', 'customer_segment', 'booking_type', 'season']


/var/folders/y8/5pmfsnf12x30hjjlc81mqm1m0000gn/T/ipykernel_42659/2941987613.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include='object').columns.tolist()


In [54]:
# ── 3b. Cast category columns ─────────────────────────────────────────────────
for col in cat_candidates:
    df[col] = df[col].astype('category')

print('✅ Category dtypes assigned.')

# ── 3c. Cast boolean-looking integer columns ──────────────────────────────────
#
# Binary 0/1 integer columns (e.g. iscanceled, ispeakseason) → boolean
BOOL_COLS = ['iscanceled', 'ispeakseason', 'isrepeatedguest']
bool_cols_present = [c for c in BOOL_COLS if c in df.columns]

for col in bool_cols_present:
    unique_vals = df[col].dropna().unique()
    if set(unique_vals).issubset({0, 1}):
        df[col] = df[col].astype('boolean')
        print(f'  {col} → boolean')

# ── 3d. Numeric columns stored as strings (coerce) ────────────────────────────
EXPECTED_NUMERIC = [
    'leadtime', 'averagedailyrate', 'totalrevenue',
    'clvproxy', 'totalofspecialrequests', 'previouscancellations'
]
for col in EXPECTED_NUMERIC:
    if col in df.columns and df[col].dtype == object:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f'  {col} → numeric (coerced)')

print()
print('Final dtype counts:')
print(df.dtypes.value_counts())

✅ Category dtypes assigned.

Final dtype counts:
int64       22
float64      8
str          2
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


---
## Step 4 · Handle Missing Values

In [55]:
# ── 4a. Full missing value audit ──────────────────────────────────────────────
missing = (
    pd.DataFrame({
        'missing_count' : df.isna().sum(),
        'missing_pct'   : (df.isna().mean() * 100).round(2),
        'dtype'         : df.dtypes
    })
    .query('missing_count > 0')
    .sort_values('missing_pct', ascending=False)
)

print(f'Columns with missing values: {len(missing)}')
display(missing)

Columns with missing values: 2


,missing_count,missing_pct,dtype
lead_time_category,6264,5.25,category
stay_duration_category,645,0.54,category


In [56]:
# ── 4c. Imputation strategy ───────────────────────────────────────────────────
#
# Strategy:
#   • >50% missing   → drop the column (too sparse to be useful)
#   • Numeric        → fill with median (robust to outliers)
#   • Categorical    → fill with mode (most frequent value)
#   • Known derivable columns (leadtimecategory, pricetier) → derive from source

DROP_THRESHOLD = 50.0  # percent

cols_to_drop = missing[missing['missing_pct'] > DROP_THRESHOLD].index.tolist()
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f'🗑️  Dropped {len(cols_to_drop)} columns (>50% missing): {cols_to_drop}')
else:
    print('No columns exceeded 50% missing threshold.')

# Re-calculate missing after drops
still_missing = df.columns[df.isna().any()].tolist()
print(f'Columns still needing imputation: {still_missing}')

No columns exceeded 50% missing threshold.
Columns still needing imputation: ['lead_time_category', 'stay_duration_category']


In [57]:
# ── 4d. Derive leadtimecategory from leadtime ─────────────────────────────────
if 'leadtimecategory' in df.columns and 'leadtime' in df.columns:
    mask = df['leadtimecategory'].isna()
    df.loc[mask, 'leadtimecategory'] = pd.cut(
        df.loc[mask, 'leadtime'],
        bins=[-1, 7, 30, 90, 180, 365, float('inf')],
        labels=['Same Week', 'Short', 'Medium', 'Long', 'Very Long', 'Extreme']
    )
    print(f'✅ leadtimecategory: filled {mask.sum():,} missing values from leadtime.')

# ── 4e. Derive pricetier from averagedailyrate ────────────────────────────────
if 'pricetier' in df.columns and 'averagedailyrate' in df.columns:
    mask = df['pricetier'].isna()
    df.loc[mask, 'pricetier'] = pd.qcut(
        df.loc[mask, 'averagedailyrate'],
        q=4,
        labels=['Budget', 'Economy', 'Standard', 'Premium'],
        duplicates='drop'
    )
    print(f'✅ pricetier: filled {mask.sum():,} missing values from averagedailyrate.')

In [58]:
# ── 4f. Fill remaining numeric → median, categorical → mode ───────────────────
imputed_log = []

for col in df.columns[df.isna().any()]:
    n_missing = df[col].isna().sum()
    col_dtype = df[col].dtype

    if pd.api.types.is_numeric_dtype(col_dtype):
        fill_val = df[col].median()
        strategy = 'median'
    else:
        mode_result = df[col].mode()
        fill_val = mode_result.iloc[0] if not mode_result.empty else 'Unknown'
        strategy = 'mode'

    df[col] = df[col].fillna(fill_val)
    imputed_log.append({'column': col, 'missing_filled': n_missing,
                        'strategy': strategy, 'fill_value': str(fill_val)})

imputed_df = pd.DataFrame(imputed_log)
if not imputed_df.empty:
    print('Imputation log:')
    display(imputed_df)
else:
    print('✅ No further imputation needed.')

# Confirm zero missing
remaining = df.isna().sum().sum()
print(f'\nTotal missing values after imputation: {remaining}')

Imputation log:


,column,missing_filled,strategy,fill_value
0,lead_time_category,6264,mode,Medium Term (31-90d)
1,stay_duration_category,645,mode,2-3 Nights



Total missing values after imputation: 0


---
## Step 5 · Remove Duplicates

In [59]:
# ── 5a. Exact duplicate rows ──────────────────────────────────────────────────
n_before = len(df)
dup_count = df.duplicated().sum()

print(f'Duplicate rows found : {dup_count:,}  ({dup_count / n_before * 100:.2f}%)')

if dup_count > 0:
    # Show a sample of duplicates before dropping
    print('Sample of duplicate rows:')
    display(df[df.duplicated(keep=False)].head(6))

    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'\n✅ Dropped {dup_count:,} duplicates.  New shape: {df.shape}')
else:
    print('✅ No duplicate rows found.')

Duplicate rows found : 31,990  (26.84%)
Sample of duplicate rows:


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,average_daily_rate,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,meal_type,country_name,total_nights,total_guests,total_revenue,has_children,is_weekend_stay,has_special_requests,is_direct_booking,lead_time_category,stay_duration_category,price_tier,customer_segment,booking_type,season,is_peak_season,revenue_per_guest,average_night_value,clv_proxy
4,Resort Hotel,0,14.00,2015,July,27,1,0,2,2,0.00,0,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,0,Transient,98.00,0,1,Check-Out,2015-07-03,Bed & Breakfast,United Kingdom,2,2.00,196.00,0,0,1,0,Short Term (8-30d),2-3 Nights,Standard,Non-Family,Standard Booking,Summer,1,98.00,49.00,294.00
5,Resort Hotel,0,14.00,2015,July,27,1,0,2,2,0.00,0,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,0,Transient,98.00,0,1,Check-Out,2015-07-03,Bed & Breakfast,United Kingdom,2,2.00,196.00,0,0,1,0,Short Term (8-30d),2-3 Nights,Standard,Non-Family,Standard Booking,Summer,1,98.00,49.00,294.00
21,Resort Hotel,0,72.00,2015,July,27,1,2,4,2,0.00,0,PRT,Direct,Direct,0,0,0,A,A,1,No Deposit,250,0,Transient,84.67,0,1,Check-Out,2015-07-07,Bed & Breakfast,Portugal,6,2.00,508.02,0,1,1,0,Medium Term (31-90d),4-7 Nights,Economy,Non-Family,Standard Booking,Summer,1,254.01,42.34,762.03
22,Resort Hotel,0,72.00,2015,July,27,1,2,4,2,0.00,0,PRT,Direct,Direct,0,0,0,A,A,1,No Deposit,250,0,Transient,84.67,0,1,Check-Out,2015-07-07,Bed & Breakfast,Portugal,6,2.00,508.02,0,1,1,0,Medium Term (31-90d),4-7 Nights,Economy,Non-Family,Standard Booking,Summer,1,254.01,42.34,762.03
39,Resort Hotel,0,70.00,2015,July,27,2,2,3,2,0.00,0,ROU,Direct,Direct,0,0,0,E,E,0,No Deposit,250,0,Transient,137.00,0,1,Check-Out,2015-07-07,Half Board (2 Meals),Other Countries,5,2.00,685.00,0,1,1,0,Medium Term (31-90d),4-7 Nights,Premium,Non-Family,Standard Booking,Summer,1,342.50,68.50,1027.50
43,Resort Hotel,0,70.00,2015,July,27,2,2,3,2,0.00,0,ROU,Direct,Direct,0,0,0,E,E,0,No Deposit,250,0,Transient,137.00,0,1,Check-Out,2015-07-07,Half Board (2 Meals),Other Countries,5,2.00,685.00,0,1,1,0,Medium Term (31-90d),4-7 Nights,Premium,Non-Family,Standard Booking,Summer,1,342.50,68.50,1027.50



✅ Dropped 31,990 duplicates.  New shape: (87219, 49)


---
## Step 6 · Outlier Detection & Capping

In [60]:
# ── 6a. IQR-based outlier detection ──────────────────────────────────────────
#
# We use the IQR method to identify outliers in key numeric columns.
# Strategy: CAP (Winsorise) at the 1st and 99th percentile.
# This preserves all rows while limiting the influence of extreme values.

OUTLIER_COLS = [
    'lead_time', 'average_daily_rate', 'total_revenue',
    'clv_proxy', 'previous_cancellations', 'total_of_special_requests'
]
outlier_cols_present = [c for c in OUTLIER_COLS if c in df.columns]

outlier_report = []

for col in outlier_cols_present:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_iqr = Q1 - 1.5 * IQR
    upper_iqr = Q3 + 1.5 * IQR

    n_outliers = ((df[col] < lower_iqr) | (df[col] > upper_iqr)).sum()
    p1  = df[col].quantile(0.01)
    p99 = df[col].quantile(0.99)

    outlier_report.append({
        'column'    : col,
        'iqr_outliers': n_outliers,
        'iqr_pct'   : round(n_outliers / len(df) * 100, 2),
        'cap_lower' : round(p1, 2),
        'cap_upper' : round(p99, 2),
        'original_min': round(df[col].min(), 2),
        'original_max': round(df[col].max(), 2)
    })

print('Outlier detection report (IQR method):')
display(pd.DataFrame(outlier_report))

Outlier detection report (IQR method):


,column,iqr_outliers,iqr_pct,cap_lower,cap_upper,original_min,original_max
0,lead_time,2394,2.74,0.00,347.00,0.00,444.00
1,average_daily_rate,2505,2.87,0.00,252.00,0.00,252.00
2,total_revenue,5131,5.88,0.00,1764.00,0.00,7590.00
3,clv_proxy,5233,6.00,0.00,2424.00,0.00,11385.00
4,previous_cancellations,1681,1.93,0.00,1.00,0.00,26.00
5,total_of_special_requests,2668,3.06,0.00,3.00,0.00,5.00


In [61]:
# ── 6b. Apply Winsorisation (1st–99th percentile capping) ─────────────────────
for col in outlier_cols_present:
    p1  = df[col].quantile(0.01)
    p99 = df[col].quantile(0.99)
    n_capped_low  = (df[col] < p1).sum()
    n_capped_high = (df[col] > p99).sum()
    df[col] = df[col].clip(lower=p1, upper=p99)
    print(f'  {col:<35} capped low={n_capped_low:>4}  high={n_capped_high:>4}  '
          f'→ range now [{p1:.2f}, {p99:.2f}]')

print('\n✅ Outlier capping complete.')

  lead_time                           capped low=   0  high= 863  → range now [0.00, 347.00]
  average_daily_rate                  capped low=   0  high=   0  → range now [0.00, 252.00]
  total_revenue                       capped low=   0  high= 732  → range now [0.00, 1764.00]
  clv_proxy                           capped low=   0  high= 872  → range now [0.00, 2424.00]
  previous_cancellations              capped low=   0  high= 276  → range now [0.00, 1.00]
  total_of_special_requests           capped low=   0  high= 354  → range now [0.00, 3.00]

✅ Outlier capping complete.


---
## Step 7 · Fix Inconsistent / Invalid Values

In [62]:
# ── 7a. Negative values in columns that must be non-negative ─────────────────
NON_NEGATIVE_COLS = [
    'leadtime', 'averagedailyrate', 'totalrevenue',
    'clvproxy', 'previouscancellations', 'totalofspecialrequests'
]
nn_cols = [c for c in NON_NEGATIVE_COLS if c in df.columns]

for col in nn_cols:
    n_neg = (df[col] < 0).sum()
    if n_neg > 0:
        df[col] = df[col].clip(lower=0)
        print(f'  {col}: clipped {n_neg} negative values → 0')
    else:
        print(f'  {col}: ✅ no negatives')

In [63]:
# ── 7b. Trim whitespace in string/category columns ────────────────────────────
str_cat_cols = df.select_dtypes(include=['object', 'category']).columns

for col in str_cat_cols:
    if df[col].dtype == 'category':
        # Must remove category dtype temporarily to strip and re-categorise
        df[col] = df[col].astype(str).str.strip().str.title().astype('category')
    else:
        df[col] = df[col].str.strip().str.title()

print(f'✅ Whitespace trimmed and title-cased {len(str_cat_cols)} text columns.')

/var/folders/y8/5pmfsnf12x30hjjlc81mqm1m0000gn/T/ipykernel_42659/3662709546.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cat_cols = df.select_dtypes(include=['object', 'category']).columns


✅ Whitespace trimmed and title-cased 19 text columns.


In [64]:
# ── 7c. Validate iscanceled only contains 0/1 ─────────────────────────────────
if 'iscanceled' in df.columns:
    invalid_cancel = ~df['iscanceled'].isin([0, 1, True, False,
                                              pd.NA, np.nan])
    n_invalid = invalid_cancel.sum()
    if n_invalid > 0:
        print(f'⚠️  {n_invalid} invalid values in iscanceled → setting to NaN')
        df.loc[invalid_cancel, 'iscanceled'] = pd.NA
    else:
        print('✅ iscanceled values are all valid (0/1).')

# ── 7d. Mark and inspect any impossible ADR values ───────────────────────────
if 'averagedailyrate' in df.columns:
    zero_adr = (df['averagedailyrate'] == 0).sum()
    print(f'\nBookings with ADR = 0 (complimentary / error): {zero_adr:,}')
    # Flag them but do not remove — business may have complimentary stays
    df['is_zero_adr_flag'] = (df['averagedailyrate'] == 0).astype(int)
    print('  → Added flag column: is_zero_adr_flag')

In [65]:
# ── 7e. Check category value distributions for suspicious entries ─────────────
KEY_CATS = ['marketsegment', 'customersegment', 'deposittype', 'season']
for col in [c for c in KEY_CATS if c in df.columns]:
    print(f'\n{col} value counts:')
    print(df[col].value_counts(dropna=False).to_string())


season value counts:
season
Summer    29037
Spring    23731
Fall      18573
Winter    15878


---
## Step 8 · Final Quality Report

In [70]:

final_report = pd.DataFrame({
    'dtype'        : df.dtypes,
    'non_null'     : df.notna().sum(),
    'null_count'   : df.isna().sum(),
    'unique_values': df.nunique()
})

print('Final Column Quality Report:')
display(final_report)

Final Column Quality Report:


,dtype,non_null,null_count,unique_values
hotel,category,87219,0,2
is_canceled,int64,87219,0,2
lead_time,float64,87219,0,348
arrival_date_year,int64,87219,0,3
arrival_date_month,category,87219,0,12
arrival_date_week_number,int64,87219,0,53
arrival_date_day_of_month,int64,87219,0,31
stays_in_weekend_nights,int64,87219,0,17
stays_in_week_nights,int64,87219,0,33
adults,int64,87219,0,14


---
## Step 9 · Export Cleaned Dataset

In [69]:
# ── Export to CSV ─────────────────────────────────────────────────────────────
#
# index=False  → no unnecessary row-number column in the output file
# encoding     → UTF-8 with BOM for compatibility with Excel on Windows

df.to_csv(CLEANED_PATH, index=False, encoding='utf-8-sig')

import os
file_size_kb = os.path.getsize(CLEANED_PATH) / 1024

print('=' * 55)
print('         ✅  CLEANED DATASET EXPORTED')
print('=' * 55)
print(f'  File      : {CLEANED_PATH}')
print(f'  Rows      : {len(df):,}')
print(f'  Columns   : {df.shape[1]}')
print('=' * 55)
print()
print('First 3 rows of the cleaned dataset:')
display(df.head(3))

         ✅  CLEANED DATASET EXPORTED
  File      : /Users/musthyalasadhvik/Desktop/SectionB_G-1_StayWise/data/processed/hotel_bookings_cleaned.csv
  Rows      : 87,219
  Columns   : 49

First 3 rows of the cleaned dataset:


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,average_daily_rate,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,meal_type,country_name,total_nights,total_guests,total_revenue,has_children,is_weekend_stay,has_special_requests,is_direct_booking,lead_time_category,stay_duration_category,price_tier,customer_segment,booking_type,season,is_peak_season,revenue_per_guest,average_night_value,clv_proxy
0,Resort Hotel,0,342.00,2015,July,27,1,0,0,2,0.00,0,Prt,Direct,Direct,0,0,0,C,C,3,No Deposit,0,0,Transient,0.00,0,0,Check-Out,2015-07-01,Bed & Breakfast,Portugal,0,2.00,0.00,0,0,0,1,Very Long Term (180D+),2-3 Nights,Budget,Non-Family,Standard Booking,Summer,1,0.00,0.00,0.00
1,Resort Hotel,0,347.00,2015,July,27,1,0,0,2,0.00,0,Prt,Direct,Direct,0,0,0,C,C,4,No Deposit,0,0,Transient,0.00,0,0,Check-Out,2015-07-01,Bed & Breakfast,Portugal,0,2.00,0.00,0,0,0,1,Very Long Term (180D+),2-3 Nights,Budget,Non-Family,Standard Booking,Summer,1,0.00,0.00,0.00
2,Resort Hotel,0,7.00,2015,July,27,1,0,1,1,0.00,0,Gbr,Direct,Direct,0,0,0,A,C,0,No Deposit,0,0,Transient,75.00,0,0,Check-Out,2015-07-02,Bed & Breakfast,United Kingdom,1,1.00,75.00,0,0,0,1,Last Minute (0-7D),1 Night,Economy,Non-Family,Quick Trip,Summer,1,75.00,75.00,75.00
